# Resolution-convergence study (`resolution_check`) on a Kaggle P100 GPU

Runs the jax_constantB convergence-with-resolution study: spectrally zero-pad
a saved, already-converged state to a ladder of grids, re-polish each with
the minimum-norm Gauss-Newton / preconditioned-CG solver, and check whether
the residuals, max|&nabla;B|, and the deflection diagnostics converge as N
increases (Squire & Mallett 2022 style resolution check).

**Before running anything**: Notebook settings (right sidebar) &rarr; Accelerator &rarr; **GPU P100**.

Do not use "GPU T4 x2": the Gauss-Newton/CG solve needs `float64` (GN
tolerance 1e-10, CG tolerance 1e-26), and T4 has roughly 1/32 the
double-precision throughput of its single-precision rate -- it will likely
run this workload *slower* than plain CPU. P100 is a compute-class card with
a much better fp64:fp32 ratio (~1:2) and is the right choice here.

This notebook mirrors the structure and conventions of
`kaggle_amplitude_quest.ipynb` in this same folder: the same GPU-memory
preallocation fix, the same P100 accelerator check, the same `jax[cuda12]`
install, and the same repo clone into `/kaggle/working/jax_constantB`. See
the markdown cells below for where and why it differs -- mainly: (a) this
study *polishes an existing converged state* rather than building one from
scratch, so it needs an attached input dataset; and (b) it drives the study
loop in-kernel (importing `constantB_tools` directly) rather than via a
`!python3 ...` subprocess, so it can do per-sweep timing, a memory guard
between grids, and the resolution-vs-growth slope fit described below.

In [ ]:
# MUST run before any process (this kernel, or a later subprocess) touches
# CUDA. JAX's default is to preallocate ~75% of TOTAL device memory the moment
# it's first used, regardless of what the computation actually needs. This
# notebook only ever uses jax in the kernel process itself (no subprocess),
# but the fix is harmless and kept for parity with kaggle_amplitude_quest.ipynb
# and because the largest grid here (128,128,256) is larger than anything
# amplitude_quest.py's ladder reaches, so headroom matters more, not less.
%env XLA_PYTHON_CLIENT_PREALLOCATE=false
%env XLA_PYTHON_CLIENT_ALLOCATOR=platform

In [ ]:
import subprocess

name = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"]
).decode().strip()
print("Detected GPU:", name)

if "P100" not in name:
    print(
        "\n*** WARNING: this session is not a P100 ('%s' detected). ***\n"
        "*** This solve needs float64; T4's fp64 throughput is ~1/32 of fp32. ***\n"
        "*** Go to Settings > Accelerator > GPU P100 and restart the session. ***" % name
    )

In [ ]:
# jax[cuda12] pulls the CUDA-enabled jaxlib PJRT plugin from PyPI; it only
# needs a recent-enough NVIDIA driver (already present on Kaggle's P100
# image), not a matching system CUDA toolkit install.
#
# NOTE: pip will likely print a wall of red dependency-conflict text here
# (protobuf, numpy, etc. pinned differently by Kaggle's preinstalled
# TensorFlow/PyTorch). Those are warnings, not failures -- the install still
# succeeds. What matters is the next cell's check passing.
!pip install -q -U "jax[cuda12]"

In [ ]:
import jax
jax.config.update("jax_enable_x64", True)  # constantB_tools.py also sets this on import;
                                            # done here too so this check is meaningful standalone.
                                            # MUST happen before any other jax usage: float64
                                            # is not optional for this solve (GN tol 1e-10, CG
                                            # tol 1e-26).
import jax.numpy as jnp

print("jax version:", jax.__version__)
print("devices:", jax.devices())
print("x64 check (expect float64):", jnp.array(1.0).dtype)

assert jax.devices()[0].platform == "gpu", (
    "JAX is not seeing a GPU -- re-check the accelerator setting and the pip install above."
)

# This study always polishes with the PRECONDITIONED CG path (pcg=True):
# essential at the larger grids in GRIDS below, where the unpreconditioned
# JJ^T condition number grows like k_max^2 and a fixed CG budget solves a
# shrinking fraction of the residual per sweep. See PCG in the parameters
# cell and constantB_tools._precond.
print("This notebook always calls the solver with pcg=True (preconditioned CG).")

If this cell has already run once with GPU memory OOM errors before the
preallocation fix above was in place, **restart the session** (Run > Restart
Session) before continuing -- the kernel's own `jax.devices()` call here
holds whatever CUDA memory it claimed for the life of the kernel process, and
a stale claim from an earlier attempt will starve the study loop further
down.

In [ ]:
import os, sys

REPO_DIR = "/kaggle/working/jax_constantB"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/alfredmallet/jax_constantB.git {REPO_DIR}
else:
    print("repo already present at", REPO_DIR, "-- skipping clone")

# constantB_tools.py (jax port) holds the jit-compiled Solver/gn/CG machinery
# and the un-jitted zero_pad/load_state/save_state helpers. Imported directly
# in-kernel here (rather than run as a `!python3 resolution_check.py ...`
# subprocess, as kaggle_amplitude_quest.ipynb does for amplitude_quest.py)
# because this notebook needs fine-grained control the standalone script
# doesn't expose: per-sweep timing, a memory guard between grids, a wider
# GRIDS ladder, and the log-log slope fit at the end. Same underlying solver,
# same clone convention -- just driven from the kernel instead of a
# subprocess.
sys.path.insert(0, REPO_DIR)
from constantB_tools import (Solver, zero_pad, load_state, save_state,
                              numpy_wavenumbers, numpy_dif)
print("constantB_tools imported from", REPO_DIR)

## Physics context: resolution convergence vs. Hoelder growth

This study distinguishes two possible behaviours of max|&nabla;B| as the grid
is refined:

- **Resolution-converging** (smooth solution): max|&nabla;B| approaches an
  N-independent plateau as N increases -- the field is fully resolved and the
  observed gradients are physical, not a numerical artifact.
- **Power-law growth**, max|&nabla;B| ~ N^(1-&alpha;): a genuine non-smooth
  feature of the solution with fractional Hoelder regularity C^{0,&alpha;}.
  Refining the grid does not converge here; each refinement just resolves
  ever-finer structure at the same rate.

On the **plain** (unweighted, minimum-L2-norm) continuation branch, the
eps=0.98 switchback state showed max|&nabla;B| ~ N^(1/3) (&alpha; = 2/3): a
genuine finite-regularity feature of that branch, not a numerical artifact
(see `amplitude_quest.py`'s Sobolev-weighted-step discussion). The
**Sobolev-weighted smooth-branch** state (`quest_smooth1.npz`, the default
`STATE_FILE` below) took the smoothest available minimum-norm correction at
every continuation step and is expected, on that basis, to CONVERGE (slope
&asymp; 0) rather than grow with N.

The final cell of this notebook fits the log-log slope of max|&nabla;B| vs N
across all completed grids and reports the implied &alpha; = 1 &minus;
slope, to make that distinction quantitative rather than just eyeballing the
plot.

## Dataset assumptions -- read before running

Unlike `kaggle_amplitude_quest.ipynb` -- which starts a fresh continuation
from a hand-built seed and therefore needs no external input dataset -- this
study **polishes an already-converged state file** at successively finer
grids, and that starting state (`quest_smooth1.npz`, `mlstate_fine.npz`, or
`quest.npz`) is itself the multi-hour output of a prior amplitude-quest
continuation run. It is data, not code: it is not part of the `jax_constantB`
git repo cloned above, and `kaggle_amplitude_quest.ipynb` sets no
`/kaggle/input/...` convention for it (that notebook never reads a state file
from input -- it always initialises fresh).

**You must attach a Kaggle Dataset containing these `.npz` state files to
this notebook** (Notebook &rarr; Add Data &rarr; Upload/select your dataset)
before running. The default `STATE_FILE` in the parameters cell below
assumes that dataset is attached under the slug **`jax-constantb-states`**,
i.e. the files land at `/kaggle/input/jax-constantb-states/*.npz`.

**Check the actual slug Kaggle assigns your attached dataset** (visible in
the Data pane on the right, or in the path it shows you) and edit
`STATE_FILE` to match if it differs -- this is the one assumption in this
notebook that is specific to your Kaggle account/dataset and cannot be
inferred from the cloned repo.

## Resuming across grids and sessions

Like `amplitude_quest.py`, this study is resumable, but at grid granularity:
the study loop checks `resolution_check.csv` for `(Nx,Ny,Nz)` tuples already
recorded and skips them, so re-running the notebook (in this session or a
fresh one) continues from the next un-finished grid in `GRIDS` rather than
redoing finished ones.

`resolution_check.csv`, the per-grid `state_res{Nx}x{Nz}.npz` files, and
`fig_resolution_check.png` are all written under `/kaggle/working/`, so
**Save Version** captures them. `/kaggle/working` does **not** persist across
fresh sessions by itself: to resume a run started in an earlier session,
1. Save Version on the session that has progress you want to keep.
2. In the new session, add that previous version's output as an input dataset.
3. Copy `resolution_check.csv` (and whichever `state_res*.npz` you want to
   keep polishing from) into `/kaggle/working/` before running the study
   cell -- it will then pick up exactly where it left off.

## Dealias (Galerkin 2/3-rule) mode, and what it changes about this study

`DEALIAS` in the parameters cell below switches the underlying
`constantB_tools.py` solver from plain pointwise collocation to the strict
Galerkin 2/3-rule formulation (`Solver(shape, dealias=True)`): every field is
truncated to the retained band |k_i| < N_i/3 (strict), and because the only
nonlinearity here is quadratic, Orszag's argument makes the retained-band
projection of every product EXACT on the unpadded grid -- no aliasing
corruption anywhere in the solve. **Default is `True` for new runs.**

**With Galerkin dealiasing, the discarded-band content of `|B|^2-1`
(`Solver.tail_norm`, recorded below as `tail_rms`/`tail_max`) is the primary
regularity diagnostic, largely replacing the cross-grid refine/polish
protocol this study otherwise relies on for resolution honesty:**
- the tail is a *single-grid* honest measure of the true continuum
  constraint violation (by power-preserving alias folding, it equals the
  discarded spectral tail of a retained-band solution exactly);
- **spectral (faster than any power law) decay of the tail with N** signals
  a smooth branch -- the field is fully resolved at that N, no further
  refinement changes the physics;
- **algebraic (power-law) decay of the tail with N** signals a genuine
  finite-regularity feature: fitting `tail ~ N^(-beta)` gives a Hoelder
  exponent (the same log-log fit already done for `max|gradB|` vs N below is
  repeated for the tail when `DEALIAS=True`).

The zero-pad incoming-residual check (`S.residual(B)` printed immediately
after each zero-pad, *before* polishing) is retained as a cross-check either
way: it is still the right way to catch a badly mismatched carrier/seed or a
corrupted state file, independent of collocation vs. Galerkin.

**Old CSVs/states in this repo (`resolution_check.csv`, `state_res*.npz`)
were produced with plain collocation (`dealias=False`).** Mixing an old,
collocation-produced CSV with new Galerkin rows breaks the resume logic
below silently (different residual semantics under the same column names,
and the new `tail_rms`/`tail_max` columns don't exist in the old schema) --
`CSV_PATH` in the parameters cell therefore points at a **different
filename**, `resolution_check_galerkin.csv`, whenever `DEALIAS=True`, so the
two schemas can never collide mid-resume.

**Protocol warning (from CPU pilot runs):** the tail-vs-N ladder must ASCEND from the state's native resolution. Down-truncating a rough state destroys its fine structure and the Galerkin re-solve can then converge to a DIFFERENT, smoother nearby solution (observed: the eps=0.98 rough state truncated to 32^2x64 relaxed to a small-tail smooth neighbour). Descending points measure branch-hopping, not regularity; only native-and-upward points are meaningful for rough states.

In [ ]:
# ---- Study parameters ------------------------------------------------------
# Default: the smooth (Sobolev-weighted) branch state, expected on physical
# grounds to be resolution-CONVERGENT (see physics-context markdown above).
STATE_FILE = "/kaggle/input/jax-constantb-states/quest_smooth1.npz"
# STATE_FILE = "/kaggle/input/jax-constantb-states/mlstate_fine.npz"  # plain eps=0.98 branch: N^(1/3) growth expected
# STATE_FILE = "/kaggle/input/jax-constantb-states/quest.npz"         # raw (unweighted) continuation state

GRIDS  = [(32, 32, 64), (48, 48, 96), (64, 64, 128), (96, 96, 192), (128, 128, 256)]
SWEEPS = 40      # GN sweeps per grid (each sweep = one CG solve + minimum-norm update)
CGIT   = 600     # CG iterations per sweep
TARGET = 1e-8    # stop polishing early once max residual < TARGET
PCG    = True    # preconditioned CG -- essential at large N (see constantB_tools._precond)
assert PCG, "this study is designed around the preconditioned-CG path; see resolution_check.py"

# Galerkin 2/3-rule dealiasing (see markdown above). Default True for new runs;
# set False only to reproduce the original collocation behaviour against an
# existing collocation-produced resolution_check.csv/state_res*.npz.
DEALIAS = True

# DEALIAS flips the CSV schema (adds tail_rms/tail_max) and the meaning of the
# residual columns, so it gets its own filename -- old and new runs can never
# collide mid-resume.
CSV_PATH      = ("/kaggle/working/resolution_check_galerkin.csv" if DEALIAS
                  else "/kaggle/working/resolution_check.csv")
FIG_PATH      = "/kaggle/working/fig_resolution_check.png"
STATE_OUT_DIR = "/kaggle/working"

print("STATE_FILE:", STATE_FILE)
print("GRIDS:", GRIDS)
print("SWEEPS:", SWEEPS, " CGIT:", CGIT, " TARGET:", TARGET, " PCG:", PCG, " DEALIAS:", DEALIAS)
print("CSV_PATH:", CSV_PATH)

In [ ]:
import gc, time, csv
import numpy as np


def gc_sweep():
    """Force an immediate garbage-collection pass. CPython refcounting
    means `del`ing the *caller's* own names for a finished grid's field and
    Solver (done in the study loop below, right before this call) is what
    actually drops their refcount to zero and lets jax's platform allocator
    release the underlying CUDA buffers; this just forces collection rather
    than waiting on it, so the next (larger) grid's allocation doesn't have
    to contend with a still-cached previous grid.

    Memory guard context: at (128,128,256) the field itself is only
    3 * 4.2M float64 ~= 100MB, but the CG workspace (R, Z, p, JTp, A, dB, plus
    the (3,Nx,Ny,Nz) wavenumber array K held by the Solver) multiplies that by
    roughly 15-20x -- still comfortable on a 16GB P100, but there is no reason
    to let two grids' workspaces overlap in device memory when they don't
    need to."""
    gc.collect()


def diagnostics(B, grid):
    """Host-numpy diagnostics on the polished field (one-shot per grid --
    exactly as constantB_tools.py / resolution_check.py: no benefit from
    jit)."""
    B = np.asarray(B)
    nrm = np.sqrt((B ** 2).sum(0))
    Bbar = B.mean(axis=(1, 2, 3)); nb = np.linalg.norm(Bbar)
    cosM = np.clip((B * Bbar[:, None, None, None]).sum(0) / (nrm * nb), -1, 1)
    defl = np.degrees(np.arccos(cosM))
    K = numpy_wavenumbers(grid)
    g2 = sum(numpy_dif(B[i], j, K) ** 2 for i in range(3) for j in range(3))
    return dict(res_norm=float(np.abs(nrm - 1).max()),
                maxgrad=float(np.sqrt(g2.max())),
                maxdefl=float(defl.max()),
                vol_rev=float((defl > 90).mean()))

In [ ]:
B0state, eps0, meta = load_state(STATE_FILE)
print(f"loaded {STATE_FILE}: eps={eps0:.3f}, base grid {B0state.shape[1:]}")

done = set()
if os.path.exists(CSV_PATH):
    with open(CSV_PATH) as f:
        for r in csv.DictReader(f):
            done.add((int(r['Nx']), int(r['Ny']), int(r['Nz'])))
    print(f"resuming: {len(done)} grid(s) already in {CSV_PATH}")

fresh = not os.path.exists(CSV_PATH)

for grid in GRIDS:
    if grid in done:
        print(f"[{grid}] already in {CSV_PATH} -- skipping")
        continue

    t0_grid = time.time()

    # 1) spectrally zero-pad the ORIGINAL loaded state to this grid (exact
    #    for the retained modes; NOT chained from the previous grid's
    #    polish, so the incoming residual measured next is the honest
    #    continuum residual of the incoming solution -- exactly as in
    #    resolution_check.py).
    B = jnp.stack([zero_pad(B0state[i], grid) for i in range(3)])
    S = Solver(grid, dealias=DEALIAS)

    # 2) HONEST incoming residual, printed BEFORE any polishing.
    r1, r2 = S.residual(B)
    r1 = np.asarray(r1); r2 = np.asarray(r2)
    print(f"[{grid}] honest incoming residual: div {np.abs(r1).max():.2e}, "
          f"|B|^2-1 {np.abs(2 * r2).max():.2e}")
    del r1, r2

    # 3) re-polish with the minimum-norm Gauss-Newton / preconditioned-CG
    #    solver until residual < TARGET or the sweep budget is exhausted.
    #    `_gn_jit` compiles ALL `sweeps` GN sweeps into one XLA program, so to
    #    get PER-SWEEP wall time (required for this study) we call it with
    #    sweeps=1 repeatedly instead of sweeps=SWEEPS once. `sweeps`/`cgit`/
    #    `tol` are traced (non-static) arguments of `_gn_jit`, so looping like
    #    this does not force a recompile beyond the first call at this grid
    #    shape -- only `pcg`/`weighted`/`verbose` are static.
    res = float(np.max(np.abs(np.asarray(S.residual(B)[0]))))
    ci = 0
    n_sweeps_run = 0
    for k in range(SWEEPS):
        t0_sweep = time.time()
        B, res, ci = S.gn(B, sweeps=1, cgit=CGIT, tol=TARGET, verbose=False, pcg=PCG)
        n_sweeps_run += 1
        dt_sweep = time.time() - t0_sweep
        print(f"    [{grid}] sweep {k}: residual {res:.2e}  cg_iters {ci}  ({dt_sweep:.2f} s)")
        if res < TARGET:
            break
    if res >= TARGET:
        print(f"    [{grid}] sweep budget ({SWEEPS}) exhausted without reaching "
              f"TARGET={TARGET:.1e} (last residual {res:.2e}) -- raise SWEEPS/CGIT "
              "for this grid if it has not plateaued.")

    # 4) diagnostics + CSV row
    B = np.asarray(B)
    d = diagnostics(B, grid)
    dt_grid = (time.time() - t0_grid) / 60.0
    row = dict(Nx=grid[0], Ny=grid[1], Nz=grid[2],
               res_div=float(np.abs(np.asarray(S.residual(B)[0])).max()),
               res_norm=d['res_norm'], maxgrad=d['maxgrad'], maxdefl=d['maxdefl'],
               vol_rev=d['vol_rev'], sweeps_run=n_sweeps_run, cg_iters_last=ci,
               minutes=dt_grid)
    if DEALIAS:
        # discarded-band content of |B|^2-1 -- the honest single-grid
        # convergence diagnostic in Galerkin mode (see markdown above and
        # constantB_tools.py's Solver.tail_norm).
        tail_rms, tail_max = S.tail_norm(B)
        row['tail_rms'] = tail_rms
        row['tail_max'] = tail_max
    print(f"[{grid}] polished: | |B|-1 | {row['res_norm']:.2e}  div {row['res_div']:.2e}  "
          f"maxgrad {row['maxgrad']:.3f}  maxdefl {row['maxdefl']:.2f}  "
          f"vol>90 {100 * row['vol_rev']:.2f}%  sweeps {n_sweeps_run}" +
          (f"  tail(rms/max) {row['tail_rms']:.2e}/{row['tail_max']:.2e}" if DEALIAS else "") +
          f"  ({dt_grid:.1f} min)")

    with open(CSV_PATH, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(row))
        if fresh:
            w.writeheader(); fresh = False
        w.writerow(row)

    np.savez(os.path.join(STATE_OUT_DIR, f"state_res{grid[0]}x{grid[2]}.npz"),
             B=B, eps=eps0, **meta)

    # 5) memory guard: drop this grid's field + Solver (which holds the
    #    (3,Nx,Ny,Nz) wavenumber array K and K2) before the next, larger grid
    #    allocates its own field + CG workspace.
    del B, S, d, row
    gc_sweep()

print("study loop done.")

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Reload cumulative rows from CSV (covers both this run's new grids and any
# grids finished in a previous, resumed session) so the plot and the slope
# fit reflect everything completed so far, not just this run's additions.
all_rows = []
with open(CSV_PATH) as f:
    for r in csv.DictReader(f):
        row = {}
        for k, v in r.items():
            row[k] = int(v) if k in ('Nx', 'Ny', 'Nz', 'sweeps_run', 'cg_iters_last') else float(v)
        all_rows.append(row)
all_rows.sort(key=lambda r: r['Nx'])
has_tail = DEALIAS and len(all_rows) > 0 and 'tail_rms' in all_rows[0]

N = [r['Nx'] for r in all_rows]
n_panels = 4 if has_tail else 3
fig, ax = plt.subplots(1, n_panels, figsize=(11 if not has_tail else 14.5, 3.2))
ax[0].semilogy(N, [r['res_norm'] for r in all_rows], 'o-', label=r'max$||B|-1|$')
ax[0].semilogy(N, [r['res_div'] for r in all_rows], 's-', label=r'max$|\nabla\cdot B|$')
ax[0].set_xlabel(r'$N_x$'); ax[0].legend(); ax[0].set_title('constraint residuals')
ax[1].plot(N, [r['maxgrad'] for r in all_rows], 'o-')
ax[1].set_xlabel(r'$N_x$'); ax[1].set_title(r'max$|\nabla B|_F$')
ax[2].plot(N, [r['maxdefl'] for r in all_rows], 'o-')
ax[2].set_xlabel(r'$N_x$'); ax[2].set_title('max deflection (deg)')
if has_tail:
    ax[3].semilogy(N, [r['tail_rms'] for r in all_rows], 'o-', label='rms')
    ax[3].semilogy(N, [r['tail_max'] for r in all_rows], 's-', label='max')
    ax[3].set_xlabel(r'$N_x$'); ax[3].legend()
    ax[3].set_title('discarded-band tail of |B|^2-1\n(Galerkin regularity diagnostic)', fontsize=9)
for a in ax:
    a.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_PATH, dpi=200)
print("wrote", FIG_PATH)

# ---- fitted log-log slope of max|gradB| vs N, and implied Hoelder exponent ----
if len(all_rows) >= 2:
    logN = np.log(N)
    logG = np.log([r['maxgrad'] for r in all_rows])
    slope, intercept = np.polyfit(logN, logG, 1)
    alpha = 1.0 - slope
    print(f"fitted log-log slope of max|gradB| vs N across {len(all_rows)} completed grid(s): {slope:.3f}")
    print(f"implied Hoelder exponent alpha = 1 - slope = {alpha:.3f}")
    if slope < 0.05:
        print("  -> max|gradB| is resolution-CONVERGING (slope ~ 0): consistent with a smooth solution.")
    else:
        print(f"  -> max|gradB| grows ~ N^{slope:.2f}: power-law growth signature "
              f"(fractional Hoelder regularity C^0,alpha with alpha~{alpha:.2f}); "
              "cf. the plain eps=0.98 branch's N^(1/3) growth (alpha=2/3).")
else:
    print("need >= 2 completed grids to fit a log-log slope.")
    # robustness: successive pairwise slopes (asymptotic regime check).
    # A genuine power law shows STABLE pairwise slopes across refinements;
    # a drifting sequence means the small grids are pre-asymptotic and the
    # global fit should be restricted to the largest grids.
    if len(all_rows) >= 3 and 'logG' in dir():
        pw = np.diff(logG)/np.diff(logN)
        print("pairwise slopes:", ", ".join(f"{s:.3f}" for s in pw),
              " -> pairwise alpha:", ", ".join(f"{1-s:.3f}" for s in pw))


# ---- (Galerkin mode only) fitted log-log slope of the discarded-band tail
# vs N, and implied decay class. This is the PRIMARY regularity diagnostic
# in dealias mode (see the "Dealias (Galerkin 2/3-rule) mode" markdown
# above): unlike max|gradB| (a pointwise quantity that can plateau or grow
# for reasons unrelated to resolution), the tail is by construction the
# discarded spectral content, so its decay with N directly measures whether
# the solution is fully resolved (spectral/faster-than-any-power decay) or
# has a genuine finite-regularity feature (algebraic/power-law decay).
if has_tail and len(all_rows) >= 2:
    logN = np.log(N)
    logT = np.log([r['tail_rms'] for r in all_rows])
    tslope, tintercept = np.polyfit(logN, logT, 1)
    print(f"fitted log-log slope of tail_rms vs N across {len(all_rows)} completed grid(s): {tslope:.3f}")
    if tslope < -1.5:
        print(f"  -> tail decays steeply (slope {tslope:.2f} < -1.5): consistent with a SMOOTH branch "
              "(spectral-like decay; each refinement mainly reduces round-off/near-machine-eps content).")
    else:
        beta = -tslope
        print(f"  -> tail decays ~ N^{tslope:.2f} (Hoelder-type exponent beta~{beta:.2f}): "
              "algebraic decay signature -- a genuine finite-regularity feature that refinement "
              "resolves at a fixed rate rather than converging away. Cross-check against the "
              "max|gradB| slope above and the zero-pad incoming-residual prints per grid.")
    if len(all_rows) >= 3:
        pw = np.diff(logT) / np.diff(logN)
        print("pairwise tail_rms slopes:", ", ".join(f"{s:.3f}" for s in pw))
elif DEALIAS:
    print("need >= 2 completed grids with tail_rms/tail_max to fit the tail-vs-N slope "
          "(DEALIAS=True but CSV_PATH has < 2 rows so far, or predates the tail columns).")


Edit `GRIDS`/`SWEEPS`/`CGIT` in the parameters cell to taste -- e.g. drop the
largest grid or shrink `SWEEPS` for a smoke test before committing a full run
to your 30-hour/week GPU quota.

Expected per-grid wall time on a P100 (rough, and dependent on how close the
incoming state already is to converged at that grid): a few seconds at
(32,32,64)/(48,48,96); on the order of a minute at (64,64,128); several
minutes at (96,96,192); and roughly ten-plus minutes at (128,128,256), most
of which is the one-time XLA compilation for that grid shape rather than the
CG iterations themselves once compiled.